
# Notebook 1 - Integração, validação e camada de dados

Este notebook mostra como o projeto integra múltiplas fontes e gera a base consolidada usada pela aplicação Flask e pelos assets do Power BI.

## O que você vai ver
1. leitura das fontes enterprise
2. padronização de colunas
3. integração
4. validação de qualidade
5. gravação da base consolidada


In [1]:

import pandas as pd
import sqlite3
import json
from pathlib import Path

BASE = Path("..").resolve()
RAW = BASE / "data" / "raw_enterprise"
PROCESSED = BASE / "data" / "processed"


## Etapa 1 - Ler as fontes

In [2]:

clientes = pd.read_csv(RAW / "clientes.csv").rename(columns={"nome": "cliente_nome"})
agencias = pd.read_csv(RAW / "agencias.csv")
produtos = pd.read_excel(RAW / "produtos.xlsx").rename(columns={"nome":"produto_nome","tipo":"produto_tipo"})

conn = sqlite3.connect(RAW / "contratos.db")
contratos = pd.read_sql("select * from contratos", conn)
conn.close()
contratos = contratos.rename(columns={"valor":"contrato_valor","data":"contrato_data"})

with open(RAW / "transacoes.json", "r", encoding="utf-8") as f:
    transacoes = pd.DataFrame(json.load(f))
transacoes = transacoes.rename(columns={"valor":"transacao_valor","tipo":"transacao_tipo","data":"transacao_data"})

clientes.head()


,id_cliente,cliente_nome,cpf,email,telefone,endereco
0,1,Srta. Rafaela Silva,354.729.860-10,da-pazbruna@example.org,+55 (071) 0213-7256,"Alameda Maria Julia Almeida, 76, Trevo, 35772-..."
1,2,Gabriel Lopes,497.168.235-09,camargoravi@example.com,+55 (021) 1039 8784,"Viaduto Novais, 99, Vila Jardim Leblon, 933198..."
2,3,Olívia Rezende,934.125.806-51,calebrocha@example.com,+55 (071) 0815 7694,"Trevo Felipe Rodrigues, 63, Alpes, 32392-637 S..."
3,4,Amanda Cardoso,946.013.572-25,gfonseca@example.com,21 1292-6568,"Núcleo Camargo, Vila Rica, 23507-700 Alves Ale..."
4,5,Alexia Farias,610.854.392-24,luiz-fernando24@example.com,(051) 8228-7779,"Estrada Leandro Aragão, 638, Nossa Senhora De ..."


## Etapa 2 - Padronizar tipos e datas

In [3]:

contratos["contrato_data"] = pd.to_datetime(contratos["contrato_data"], errors="coerce")
transacoes["transacao_data"] = pd.to_datetime(transacoes["transacao_data"], errors="coerce")
print(clientes.shape, agencias.shape, produtos.shape, contratos.shape, transacoes.shape)


(3000, 6) (80, 4) (40, 4) (15000, 6) (60000, 5)


## Etapa 3 - Integrar contratos com dimensões

In [4]:

base_contratos = (
    contratos.merge(clientes, on="id_cliente", how="left")
             .merge(agencias, on="id_agencia", how="left")
             .merge(produtos, on="id_produto", how="left")
)
base_contratos.head()


,id_contrato,id_cliente,id_produto,id_agencia,contrato_valor,contrato_data,cliente_nome,cpf,email,telefone,endereco,nome_agencia,cidade,estado,produto_nome,produto_tipo,taxa_juros
0,1,2990,20,52,19305.08,2025-07-06,Sr. João Miguel Pires,723.954.608-00,castroisis@example.com,+55 11 6326-8129,"Favela Cecília Montenegro, Vila Antena, 459975...",Agência 052,Cassiano,PE,Empréstimo 20,Empréstimo,0.31
1,2,1850,2,29,2246.69,2025-06-13,Kamilly Cavalcanti,564.802.379-00,srodrigues@example.com,71 3816-0388,"Conjunto Henry da Luz, 928, Granja De Freitas,...",Agência 029,Siqueira,BA,Cartão 02,Cartão,0.94
2,3,2269,3,29,30356.42,2025-08-25,Maitê Pinto,587.963.402-74,bernardoda-paz@example.com,41 8948 6777,"Rua Pereira, 10, Lourdes, 74686200 Pereira de ...",Agência 029,Siqueira,BA,Investimento 03,Investimento,2.88
3,4,242,4,5,9448.18,2025-10-08,Carlos Eduardo Gomes,197.586.203-12,rioseloah@example.org,0300-778-5534,"Quadra de Mendonça, Santo André, 90367349 Cost...",Agência 005,Santos dos Dourados,GO,Cartão 04,Cartão,3.74
4,5,1141,14,70,1992.00,2024-08-31,João Gabriel Garcia,746.382.905-74,lguerra@example.org,+55 31 5420 1567,"Recanto Monteiro, 922, Solar Do Barreiro, 9258...",Agência 070,Cunha do Norte,GO,Financiamento 14,Financiamento,4.26


## Etapa 4 - Integrar transações e criar colunas derivadas

In [5]:

base_integrada = transacoes.merge(base_contratos, on="id_contrato", how="left")
base_integrada["ano_mes"] = base_integrada["transacao_data"].dt.to_period("M").astype(str)
base_integrada["ano"] = base_integrada["transacao_data"].dt.year
base_integrada["mes"] = base_integrada["transacao_data"].dt.month
base_integrada["dia"] = base_integrada["transacao_data"].dt.day
base_integrada.head()


,id_transacao,id_contrato,transacao_data,transacao_valor,transacao_tipo,id_cliente,id_produto,id_agencia,contrato_valor,contrato_data,cliente_nome,cpf,email,telefone,endereco,nome_agencia,cidade,estado,produto_nome,produto_tipo,taxa_juros,ano_mes,ano,mes,dia
0,1,14422,2024-09-16 23:50:58,10.16,Pagamento,812,27,65,7922.34,2026-01-04,Dr. João Miguel Pacheco,395.718.620-02,barrosluiz-miguel@example.net,+55 61 4651 3067,"Conjunto de Rodrigues, 74, Vila Petropolis, 14...",Agência 065,Sales,ES,Poupança 27,Poupança,2.02,2024-09,2024,9,16
1,2,6407,2024-04-12 18:00:52,12.20,Crédito,2254,7,60,93342.75,2025-04-26,André Melo,291.853.670-95,vpires@example.org,61 4518-1355,"Fazenda Farias, 583, Cinquentenário, 27080485 ...",Agência 060,Siqueira das Pedras,MG,Poupança 07,Poupança,2.64,2024-04,2024,4,12
2,3,8557,2024-08-01 02:42:22,5.00,Débito,1928,37,1,138573.42,2025-03-29,Srta. Maya Fernandes,305.249.618-24,kfarias@example.org,+55 21 0970-5926,"Área Monteiro, 848, Lagoinha Leblon, 00432-954...",Agência 001,Farias de Araújo,BA,Financiamento 37,Financiamento,3.82,2024-08,2024,8,1
3,4,11647,2024-04-18 20:50:01,5.00,Crédito,754,14,49,131719.88,2025-12-06,Joana Vargas,859.306.217-21,gustavo-henriquepires@example.com,+55 61 4038 8577,"Lago de Cavalcante, 1, Jaqueline, 13037667 Men...",Agência 049,Castro,GO,Financiamento 14,Financiamento,4.26,2024-04,2024,4,18
4,5,3620,2025-11-26 17:56:40,33.46,Crédito,1739,8,41,10066.39,2025-08-22,Sra. Sophie Sales,645.371.980-01,antony31@example.net,+55 (071) 4565-0305,"Residencial Yago Fogaça, Indaiá, 19205-562 Bor...",Agência 041,Farias da Prata,CE,Conta Corrente 08,Conta Corrente,3.78,2025-11,2025,11,26


## Etapa 5 - Validar qualidade

In [6]:

qualidade = pd.DataFrame({
    "coluna": base_integrada.columns,
    "nulos_%": [round(base_integrada[c].isna().mean()*100, 4) for c in base_integrada.columns],
    "unicos": [int(base_integrada[c].nunique(dropna=True)) for c in base_integrada.columns],
})
qualidade.head(10)


,coluna,nulos_%,unicos
0,id_transacao,0.0,60000
1,id_contrato,0.0,14734
2,transacao_data,0.0,59977
3,transacao_valor,0.0,5634
4,transacao_tipo,0.0,5
5,id_cliente,0.0,2986
6,id_produto,0.0,40
7,id_agencia,0.0,80
8,contrato_valor,0.0,14716
9,contrato_data,0.0,822


## Etapa 6 - Salvar a base integrada

In [7]:

base_integrada.to_csv(PROCESSED / "base_integrada.csv", index=False, encoding="utf-8-sig")
print("Salva em:", PROCESSED / "base_integrada.csv")


Salva em: /mnt/data/projeto_tcc_ultra/data/processed/base_integrada.csv
